[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AI-Hypercomputer/maxtext/blob/main/src/maxtext/examples/qwen3_native_lora_demo.ipynb)

# Qwen3-0.6B Parameter-Efficient Fine-Tuning (PEFT) Demo with Native LoRA & QLoRA


## Overview

This tutorial demonstrates how to run native **Qwen3-0.6B** Parameter-Efficient Fine-Tuning (PEFT) using **LoRA** and **QLoRA** under Flax NNX in MaxText.

We cover both standard PEFT workflows:
1. **Pathway A**: Native LoRA Fine-Tuning with Qwen3 on the GSM8K dataset, starting from converted Hugging Face checkpoints.
2. **Pathway B**: Native Pre-training with Qwen3 using QLoRA, including saving, verifying, and restoring checkpoints.

## Prerequisites

Before running this notebook, make sure your environment is set up for the method you are using. Follow the [Run MaxText Python Notebooks on TPUs](https://maxtext.readthedocs.io/en/latest/guides/run_python_notebook.html) guide and complete all steps for your chosen method (Google Colab, VS Code, or Local Jupyter Lab) before proceeding.

If you run into issues, refer to the [Common Pitfalls & Debugging](https://maxtext.readthedocs.io/en/latest/guides/run_python_notebook.html#common-pitfalls-debugging) section of the guide.

In [ ]:
try:
  import google.colab
  print("Running the notebook on Google Colab")
  IN_COLAB = True
except ImportError:
  print("Running the notebook on Visual Studio or JupyterLab")
  IN_COLAB = False

## Installation: MaxText and Post-Training Dependencies

**Running the notebook on Visual Studio or JupyterLab:** Before proceeding, create a virtual environment and install the required post-training dependencies by following `Option 3: Installing [tpu-post-train]` in the [MaxText installation guide](https://maxtext.readthedocs.io/en/latest/install_maxtext.html#from-source). Once the environment is set up, ensure the notebook is running within it.

In [ ]:
if IN_COLAB:
    # Clone the MaxText repository
    !git clone https://github.com/AI-Hypercomputer/maxtext.git
    %cd maxtext

    # Install uv, a fast Python package installer
    !pip install uv
    
    # Install MaxText and post-training dependencies
    import os
    os.environ["UV_TORCH_BACKEND"]="cpu"
    !uv pip install -e .[tpu-post-train] --resolution=lowest
    !install_tpu_post_train_extra_deps

**Session restart Instructions for Colab:**
1.  Navigate to the menu at the top of the screen.
2.  Click on **Runtime**.
3.  Select **Restart session** from the dropdown menu.

You will be asked to confirm the action in a pop-up dialog. Click on **Yes**.

## Setup and Imports

In [ ]:
import os
import sys
import subprocess

## Hugging Face Hub Login

Qwen3 model checkpoints are hosted on the Hugging Face Hub. Run the cell below to log in so that you can download the base model weights.

In [ ]:
try:
    from huggingface_hub import notebook_login
    notebook_login()
except ImportError:
    from huggingface_hub import login
    login()

## Download & Convert Checkpoint from Hugging Face

We convert the Hugging Face base `Qwen/Qwen3-0.6B` checkpoint to the MaxText Orbax format.

In [ ]:
MODEL_NAME = "qwen3-0.6b"
BASE_OUTPUT_DIRECTORY = "/tmp/qwen3_sft_output"
HF_CHECKPOINT_PATH = f"{BASE_OUTPUT_DIRECTORY}/qwen3_06_checkpoint"

if not os.path.exists(HF_CHECKPOINT_PATH):
    print("Converting checkpoint from Hugging Face...")
    env = os.environ.copy()
    env["JAX_PLATFORMS"] = "cpu"
    subprocess.run([
        sys.executable, "-m", "maxtext.checkpoint_conversion.to_maxtext",
        "src/maxtext/configs/base.yml",
        f"model_name={MODEL_NAME}",
        f"base_output_directory={HF_CHECKPOINT_PATH}",
        "use_multimodal=false",
        "scan_layers=true",
        "skip_jax_distributed_system=True",
    ], env=env, check=True)
    print(f"Checkpoint successfully converted to: {HF_CHECKPOINT_PATH}/0/items")
else:
    print(f"Model checkpoint already exists at: {HF_CHECKPOINT_PATH}")

## Pathway A: Native LoRA Fine-Tuning with Qwen3

We run native LoRA Supervised Fine-Tuning (SFT) on the real `openai/gsm8k` math dataset. 

We enable LoRA by passing `lora.enable_lora=True`, which ensures that only the lightweight adapter weights are updated while the base weights remain frozen.

In [ ]:
subprocess.run([
    sys.executable, "src/maxtext/trainers/post_train/sft/train_sft_native.py",
    "src/maxtext/configs/post_train/sft.yml",
    "run_name=nnx_qwen3_sft_lora_demo",
    f"load_parameters_path={HF_CHECKPOINT_PATH}/0/items",
    f"model_name={MODEL_NAME}",
    f"base_output_directory={BASE_OUTPUT_DIRECTORY}",
    "tokenizer_path=Qwen/Qwen3-0.6B",
    "hf_path=openai/gsm8k",
    "train_split=train",
    "hf_data_dir=main",
    "train_data_columns=['question', 'answer']",
    "steps=5",
    "per_device_batch_size=1",
    "max_target_length=128",
    "learning_rate=3e-6",
    "weight_dtype=bfloat16",
    "dtype=bfloat16",
    "formatting_func_path=maxtext.input_pipeline.instruction_data_processing.math_qa_formatting",
    "formatting_func_kwargs={'template_path': 'src/maxtext/examples/chat_templates/math_qa.json'}",
    "lora.enable_lora=True",
    "lora.lora_rank=4",
    "lora.lora_alpha=8.0"
], check=True)

## Pathway B: Native LoRA Pre-training & Checkpoint Restore with QLoRA

Next, we demonstrate how to run a native Flax NNX pre-training loop with **QLoRA** enabled (`lora.enable_lora=True`, `lora.lora_weight_qtype=int8`). 

We will run 1 step to save a native checkpoint, inspect the native state dict contents, and then successfully restore the parameters to resume training.

### Step 1: Run 1 step and save the checkpoint with QLoRA enabled

In [ ]:
subprocess.run([
    sys.executable, "src/maxtext/trainers/pre_train/train.py", "src/maxtext/configs/base.yml",
    "run_name=nnx_qwen3_pretrain_qlora_demo",
    "model_name=qwen3-0.6b",
    "steps=1",
    "dataset_type=synthetic",
    "per_device_batch_size=1",
    "max_target_length=32",
    "enable_checkpointing=True",
    "checkpoint_period=1",
    "async_checkpointing=False",
    "sharding_tolerance=1.0",
    "base_output_directory=/tmp/nnx_qwen3_pretrain_qlora_checkpoint",
    "attention=dot_product",
    "weight_dtype=bfloat16",
    "dtype=bfloat16",
    "lora.enable_lora=True",
    "lora.lora_weight_qtype=int8",
    "lora.lora_tile_size=32",
    "lora.lora_rank=4",
    "lora.lora_alpha=8.0"
], check=True)

### Step 2: Verify the saved checkpoint directory exists

The native Flax NNX checkpoint saver cleanly stores our checkpoint under the `checkpoints/0/items` Orbax directory.

In [ ]:
checkpoint_dir = "/tmp/nnx_qwen3_pretrain_qlora_checkpoint/nnx_qwen3_pretrain_qlora_demo/checkpoints/0/items"
print(f"Checkpoint exists: {os.path.exists(checkpoint_dir)}")
if os.path.exists(checkpoint_dir):
    print("Checkpoint contents:", os.listdir(checkpoint_dir))

### Step 3: Resume training from the saved checkpoint

We pass the saved directory path to `load_parameters_path` to load our frozen weights and LoRA adapters, successfully resuming training up to step 2.

In [ ]:
subprocess.run([
    sys.executable, "src/maxtext/trainers/pre_train/train.py", "src/maxtext/configs/base.yml",
    "run_name=nnx_qwen3_pretrain_qlora_demo",
    "model_name=qwen3-0.6b",
    "steps=2",
    "dataset_type=synthetic",
    "per_device_batch_size=1",
    "max_target_length=32",
    "enable_checkpointing=True",
    "checkpoint_period=1",
    "async_checkpointing=False",
    "sharding_tolerance=1.0",
    "base_output_directory=/tmp/nnx_qwen3_pretrain_qlora_checkpoint",
    "load_parameters_path=/tmp/nnx_qwen3_pretrain_qlora_checkpoint/nnx_qwen3_pretrain_qlora_demo/checkpoints/0/items",
    "attention=dot_product",
    "weight_dtype=bfloat16",
    "dtype=bfloat16",
    "lora.enable_lora=True",
    "lora.lora_weight_qtype=int8",
    "lora.lora_tile_size=32",
    "lora.lora_rank=4",
    "lora.lora_alpha=8.0"
], check=True)

## Conclusion

This tutorial successfully demonstrated the end-to-end Qwen3 Parameter-Efficient Fine-Tuning (PEFT) workflow using native LoRA and QLoRA under Flax NNX.

## 📚 Learn More

- **CLI Usage**: Refer to [Native NNX LoRA and QLoRA Fine-tuning](https://github.com/AI-Hypercomputer/maxtext/blob/main/docs/tutorials/posttraining/native_lora.md) for detailed CLI orchestrations.
- **Configuration**: See `src/maxtext/configs/post_train/sft.yml` and `src/maxtext/configs/base.yml` for all available LoRA options (prefixed with `lora.`).
- **Documentation**: Check `src/maxtext/trainers/post_train/sft/train_sft_native.py` and `src/maxtext/trainers/pre_train/train.py` for native Flax NNX training and checkpointing implementations.